# Fase 2 — QLoRA em GPU
Escolha manualmente um runtime com GPU. Faça download ZIP da branch do projeto no GitHub e envie abaixo. Não envie chaves, `.env` ou dados reais. O treino usa MedQuAD em inglês e exemplos internos sintéticos; tradução/revisão clínica continuam pendentes.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, subprocess, sys, shutil
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Envie somente um ZIP do projeto')
archive = next(iter(uploaded))
root = Path('/content/hospital-project')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (root / member.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Caminho inválido no ZIP')
    z.extractall(root)
projects = [p.parent for p in root.rglob('pyproject.toml')]
if len(projects) != 1:
    raise ValueError('ZIP deve conter exatamente um projeto Python')
project = projects[0]
print(project)


## Ambiente isolado
Instala o perfil PyTorch CUDA 11.8 e as bibliotecas fixadas sem alterar o Python do kernel. Downloads podem demorar. Se a GPU do runtime não for compatível, o diagnóstico indicará; este perfil foi preparado para GPUs antigas e T4, não para toda GPU existente.

In [ ]:
venv = Path('/content/hospital-training-env')
subprocess.run([sys.executable, '-m', 'venv', str(venv)], check=True)
python = str(venv / 'bin/python')
def run(*args):
    subprocess.run([python, *args], cwd=project, check=True)
run('-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu118')
run('-m', 'pip', 'install', '-r', 'requirements/finetuning.txt')
run('-m', 'pip', 'install', '-e', '.')
run('-m', 'hospital_assistant.finetuning.cli', 'doctor')


In [ ]:
if not (project / 'data/raw/MedQuAD').exists():
    run('-m', 'hospital_assistant.cli', 'download')
run('-m', 'hospital_assistant.cli', 'prepare')
run('-m', 'hospital_assistant.cli', 'synthetic')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## Teste de cinco passos
Inspecione `runs/colab-probe/metrics.json`. O teste curto não é o treinamento final. Uma falha fica em `failure.json`; para repetir use outro nome de saída.

In [ ]:
run('-m', 'hospital_assistant.finetuning.cli', 'probe', '--output', 'runs/colab-probe')


## Treinamento e comparação
Execute depois de validar o probe. O teste reservado não deve orientar hiperparâmetros. A loss, perplexidade e F1 lexical não comprovam qualidade clínica.

In [ ]:
run('-m', 'hospital_assistant.finetuning.cli', 'train', '--output', 'runs/colab-train')


In [ ]:
run('-m', 'hospital_assistant.finetuning.cli', 'evaluate', '--run', 'runs/colab-train', '--output', 'runs/colab-comparison', '--limit', '20')


## Preservar execução
Baixe antes de encerrar o runtime. ZIP contém relatórios, configuração e adaptador, não todos os pesos base. Também pode executar após uma falha para preservar o diagnóstico.

In [ ]:
archive = shutil.make_archive('/content/phase2-results', 'zip', project, 'runs')
files.download(archive)
